In [ ]:


!pip -q install rdkit

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler


In [ ]:

SEED = 42
N_SPLITS = 5
VAL_SIZE_WITHIN_TEMP = 0.5  
RADIUS = 2
N_BITS = 2048

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

def smiles_to_morgan_fp(smiles_list, radius=2, n_bits=2048):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fps.append(np.array(fp, dtype=np.float32))
        else:
            fps.append(np.zeros(n_bits, dtype=np.float32))
    return np.array(fps, dtype=np.float32)

def build_model(random_state):
    from sklearn.neural_network import MLPClassifier
    return MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=200,
        random_state=random_state
    )

def maybe_scale(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    return X_train, X_val, X_test


def evaluate_fold(model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score)
    }, y_score


In [ ]:

df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

X = smiles_to_morgan_fp(df["smiles"], radius=RADIUS, n_bits=N_BITS)
y = df["HIV_active"].astype(int).values

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_results = []
roc_curves = []

for fold, (train_idx, temp_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== FOLD {fold} / {N_SPLITS} =====")
    set_seed(SEED + fold)

    X_train_full, y_train_full = X[train_idx], y[train_idx]
    X_temp, y_temp = X[temp_idx], y[temp_idx]

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    X_train, X_val, X_test = maybe_scale(X_train_full, X_val, X_test)

    model = build_model(random_state=SEED + fold)
    model.fit(X_train, y_train_full)

    metrics, y_score = evaluate_fold(model, X_test, y_test)
    fold_results.append({
        "fold": fold,
        "test_accuracy": metrics["accuracy"],
        "test_precision": metrics["precision"],
        "test_recall": metrics["recall"],
        "test_f1": metrics["f1"],
        "test_roc_auc": metrics["roc_auc"]
    })

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_curves.append((fold, fpr, tpr, metrics["roc_auc"]))

    print({k: round(v, 4) if isinstance(v, float) else v for k, v in fold_results[-1].items()})

results_df = pd.DataFrame(fold_results)
results_df


In [ ]:

summary_rows = []
for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    mean_val = results_df[metric].mean()
    std_val = results_df[metric].std(ddof=1)
    var_val = results_df[metric].var(ddof=1)
    summary_rows.append({
        "Metric": metric.replace("test_", "").upper(),
        "Mean": mean_val,
        "Std": std_val,
        "Variance": var_val,
        "Formatted": f"{mean_val:.4f} ± {std_val:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:

plt.figure(figsize=(8, 6))
for fold, fpr, tpr, auc_val in roc_curves:
    plt.plot(fpr, tpr, label=f"Fold {fold} (AUC={auc_val:.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("MLP ROC Curves Across 5 Folds")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

results_df.to_csv("MLP_fold_results.csv", index=False)
summary_df.to_csv("MLP_summary_results.csv", index=False)

print("Saved:")
print("- MLP_fold_results.csv")
print("- MLP_summary_results.csv")


In [ ]:

from sklearn.metrics import log_loss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


if 'df' not in globals():
    df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

if 'X' not in globals():
    X = smiles_to_morgan_fp(df["smiles"], radius=RADIUS, n_bits=N_BITS)

if 'y' not in globals():
    y = df["HIV_active"].astype(int).values


skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

loss_rows = []

for fold, (train_idx, temp_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== FOLD {fold} / {N_SPLITS} =====")

    set_seed(SEED + fold)

    X_train_full, y_train_full = X[train_idx], y[train_idx]
    X_temp, y_temp = X[temp_idx], y[temp_idx]

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    X_train, X_val, X_test = maybe_scale(X_train_full, X_val, X_test)

    model = build_model(random_state=SEED + fold)
    model.fit(X_train, y_train_full)

    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_val_proba   = model.predict_proba(X_val)[:, 1]
    y_test_proba  = model.predict_proba(X_test)[:, 1]

   
    train_loss = log_loss(y_train_full, y_train_proba)
    val_loss   = log_loss(y_val, y_val_proba)
    test_loss  = log_loss(y_test, y_test_proba)

    loss_rows.append({
        "fold": fold,
        "train_log_loss": train_loss,
        "val_log_loss": val_loss,
        "test_log_loss": test_loss
    })

    print({
        "fold": fold,
        "train_log_loss": round(train_loss, 4),
        "val_log_loss": round(val_loss, 4),
        "test_log_loss": round(test_loss, 4)
    })

loss_df = pd.DataFrame(loss_rows)
display(loss_df)

print("\nMean ± Std")
print(f"Train Log-Loss: {loss_df['train_log_loss'].mean():.4f} ± {loss_df['train_log_loss'].std(ddof=1):.4f}")
print(f"Val Log-Loss:   {loss_df['val_log_loss'].mean():.4f} ± {loss_df['val_log_loss'].std(ddof=1):.4f}")
print(f"Test Log-Loss:  {loss_df['test_log_loss'].mean():.4f} ± {loss_df['test_log_loss'].std(ddof=1):.4f}")


x = np.arange(1, len(loss_df) + 1)

plt.figure(figsize=(8, 5))
plt.plot(x, loss_df["train_log_loss"], marker='o', linewidth=2, label="Train Log-Loss")
plt.plot(x, loss_df["val_log_loss"], marker='s', linewidth=2, label="Validation Log-Loss")
plt.plot(x, loss_df["test_log_loss"], marker='^', linewidth=2, label="Test Log-Loss")


plt.xlabel("Fold")
plt.ylabel("Log-Loss")
plt.title("Train vs Validation Log-Loss Across 5 Folds")
plt.xticks(x)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.xlabel("Fold")
plt.ylabel("Log-Loss")
plt.title("MLP Log-Loss Across 5 Folds")
plt.xticks(x)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(x, loss_df["train_log_loss"], linewidth=3, label="Train")
plt.plot(x, loss_df["val_log_loss"], linewidth=3, label="Validation")
plt.plot(x, loss_df["test_log_loss"], linewidth=3, label="Test")

plt.xlabel("Fold")
plt.ylabel("Log-Loss")
plt.title("MLP Log-Loss Across 5 Folds")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(x, loss_df["val_log_loss"], linewidth=3, label="Validation Log-Loss")
plt.plot(x, loss_df["test_log_loss"], linewidth=3, label="Test Log-Loss")

plt.xlabel("Fold")
plt.ylabel("Log-Loss")
plt.title("MLP Validation vs Test Log-Loss Across 5 Folds")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(x, loss_df["val_log_loss"], linewidth=3, label="Validation")
plt.plot(x, loss_df["test_log_loss"], linewidth=3, label="Test")

plt.ylim(0.5, 0.8)   # 🔥 kritik

plt.xlabel("Fold")
plt.ylabel("Log-Loss")
plt.title("Validation vs Test Log-Loss Across 5 Folds")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(x, loss_df["val_log_loss"], linewidth=2, label="Validation")
plt.plot(x, loss_df["test_log_loss"], linewidth=2, label="Test")

plt.fill_between(x,
                 loss_df["val_log_loss"],
                 loss_df["test_log_loss"],
                 alpha=0.2)

plt.xlabel("Fold")
plt.ylabel("Log-Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.plot(x, loss_df["val_log_loss"], linewidth=3)
plt.plot(x, loss_df["test_log_loss"], linewidth=3)

In [ ]:
means = [
    loss_df["train_log_loss"].mean(),
    loss_df["val_log_loss"].mean(),
    loss_df["test_log_loss"].mean()
]

stds = [
    loss_df["train_log_loss"].std(),
    loss_df["val_log_loss"].std(),
    loss_df["test_log_loss"].std()
]

labels = ["Train", "Validation", "Test"]

plt.figure(figsize=(6,4))
plt.bar(labels, means, yerr=stds)

plt.ylabel("Log-Loss")
plt.title("MLP Log-Loss (Mean ± Std)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from scipy.interpolate import make_interp_spline

x_smooth = np.linspace(x.min(), x.max(), 100)

y_smooth = make_interp_spline(x, loss_df["val_log_loss"])(x_smooth)

plt.plot(x_smooth, y_smooth, linewidth=3)